# Pipeline para ingesta de datos en Batch


## Primero modifica la tabla patients para ver cómo se hace la ingesta incremental

INSERT INTO pacientes (paciente_id, nombre, edad, genero, email, historia_medica)
VALUES
(8, 'Laura Martínez', 29, 'F', 'laura.martinezexample.com', 'Sin antecedentes familiares'),
(9, 'Miguel García', -10, 'M', 'miguel.garcia@example.com', 'Hipertensión y obesidad'),
(10, 'Laura García', 20, 'M', 'laura.garcia@example.com', 'Hipertensión y obesidad');


## Prueba a ejecutar otra vez con otros cambios
INSERT INTO pacientes (paciente_id, nombre, edad, genero, email, historia_medica)
VALUES
(11, 'Marta Martínez', 20, 'M', 'marta.martinez@example.com', 'Hipertensión y obesidad');

UPDATE pacientes
SET nombre='Laura García', historia_medica= 'Hipertensión'
WHERE paciente_id=10;

## Crea la conexión a spark

In [1]:
from pyspark.sql import SparkSession
from delta import *
spark = (SparkSession.builder 
    .appName("BatchEstructurado") 
    .master("spark://spark-master:7077") 
    .config("spark.submit.deployMode", "client") 
         
    # Extensiones Delta 
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
         
    # Crea el catálogo como un DeltaCatalog
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
         
    .config("spark.sql.warehouse.dir", "s3a://hospital/")


    # Establecer la conexión a minio
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") 
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123") 
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") 
    .config("spark.hadoop.fs.s3a.path.style.access", "true") 
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") 
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")   
    .config("spark.jars.packages","org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.1") 
    .config("spark.jars","/opt/spark/jars/hadoop-aws-3.4.0.jar, /opt/spark/jars/aws-java-sdk-bundle-2.23.19.jar, /opt/spark/jars/delta-spark_2.13-4.0.0.jar, /opt/spark/jars/delta-storage-4.0.0.jar, /opt/spark/jars/antlr4-runtime-4.13.1.jar, /opt/spark/jars/spark-sql-kafka-0-10_2.13-4.0.1.jar, /opt/spark/jars/kafka-clients-3.9.1.jar, /opt/spark/jars/mssql-jdbc-12.10.2.jre11.jar") 


    .getOrCreate())


print("SparkSession creada")

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jovyan/.ivy2.5.2/cache
The jars for the packages stored in: /home/jovyan/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-97f44bbf-7eca-4131-9289-24a9b21dc5ba;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.0.1 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found org.slf4j#slf4j-api;2.0.16 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.1 in central
	found org.apache.hadoop#hadoop-client-api;3.4.1 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala

SparkSession creada


## 1. Leer cambios (Change Data Capture) CDC desde Azure SQL
LSN significa "Log Sequence Number" (Número de Secuencia de Registro) en Azure SQL, y es un valor numérico que identifica y ordena secuencialmente las entradas en el registro de transacciones de la base de datos, permitiendo rastrear los cambios que ocurren en la misma
Si es la primera vez que se lee la tabla patients (LSN = NULL), ha de traer todo el CDC. Si ya hay LSN, solo ha de traer los cambios nuevos.

In [2]:
jdbc_url = (
    "jdbc:sqlserver://sqlserver:1433;"
    "databaseName=HOSPITAL;"
    "encrypt=true;"
    "trustServerCertificate=true;"
)

connection_properties = {
    "user": "sa", # tu usuario
    "password": "Password1234%", # tu password
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}


checkpoint_lsn = (spark.read.format("delta").load("s3a://hospital/bronze/pacientes_checkpoint")
                      .selectExpr("max(last_lsn)")
                      .first()[0])

print(checkpoint_lsn)

if checkpoint_lsn is None:
    cdc_table = "(SELECT * FROM cdc.dbo_pacientes_CT) AS cdc_src"
else:
    cdc_table = f"(SELECT * FROM cdc.dbo_pacientes_CT WHERE __$start_lsn > 0x{checkpoint_lsn.hex()}) AS cdc_src"

# Leer CDC:
df_cdc = spark.read.jdbc(
    url=jdbc_url,
    table=cdc_table,
    properties=connection_properties
)

df_cdc.show()



25/12/24 13:20:15 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
25/12/24 13:20:29 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


bytearray(b'\x00\x00\x003\x00\x0098\x00\x05')


+--------------------+----------+--------------------+------------+--------------+-----------+--------------+----+------+--------------------+--------------------+-------------+
|        __$start_lsn|__$end_lsn|           __$seqval|__$operation|__$update_mask|paciente_id|        nombre|edad|genero|               email|     historia_medica|__$command_id|
+--------------------+----------+--------------------+------------+--------------+-----------+--------------+----+------+--------------------+--------------------+-------------+
|[00 00 00 33 00 0...|      NULL|[00 00 00 33 00 0...|           2|          [3F]|         11|Marta Martínez|  20|     M|marta.martinez@ex...|Hipertensión y ob...|            1|
|[00 00 00 33 00 0...|      NULL|[00 00 00 33 00 0...|           3|          [20]|         10|  Laura García|  20|     M|laura.garcia@exam...|Hipertensión y ob...|            1|
|[00 00 00 33 00 0...|      NULL|[00 00 00 33 00 0...|           4|          [20]|         10|  Laura García| 


## 2. Preparar CDC para MERGE
Operaciones:

1 → DELETE

2 → INSERT

4 → UPDATE (after)


In [3]:
from pyspark.sql import Window
from pyspark.sql.functions import col, row_number

df_cdc_no3 = (
    df_cdc
    .withColumnRenamed("__$start_lsn", "start_lsn")
    .withColumnRenamed("__$operation", "op")
    .filter(col("op") != 3)
)

# Ventana: particiona por paciente_id y ordena por start_lsn descendente
window_spec = Window.partitionBy("paciente_id").orderBy(col("start_lsn").desc())

df_cdc_prepared = (
    df_cdc_no3
    .withColumnRenamed("__$start_lsn", "start_lsn")
    .withColumnRenamed("__$operation", "op")
    .withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") == 1)
    .drop("rn")
)



df_cdc_prepared.show()

+--------------------+----------+--------------------+---+--------------+-----------+--------------+----+------+--------------------+--------------------+-------------+
|           start_lsn|__$end_lsn|           __$seqval| op|__$update_mask|paciente_id|        nombre|edad|genero|               email|     historia_medica|__$command_id|
+--------------------+----------+--------------------+---+--------------+-----------+--------------+----+------+--------------------+--------------------+-------------+
|[00 00 00 33 00 0...|      NULL|[00 00 00 33 00 0...|  4|          [20]|         10|  Laura García|  20|     M|laura.garcia@exam...|        Hipertensión|            1|
|[00 00 00 33 00 0...|      NULL|[00 00 00 33 00 0...|  2|          [3F]|         11|Marta Martínez|  20|     M|marta.martinez@ex...|Hipertensión y ob...|            1|
+--------------------+----------+--------------------+---+--------------+-----------+--------------+----+------+--------------------+--------------------+-

## 3. MERGE incremental en Bronze (CDC)

In [4]:
from delta.tables import DeltaTable

bronze_table="s3a://hospital/bronze/pacientes"

delta_table = DeltaTable.forPath(spark, bronze_table)

# recupera los nombres de las columnas
target_cols = delta_table.toDF().columns

# actualiza los datos de pacientes ya existentes en caso de update o delete


# Hacer el MERGE
(
    delta_table.alias("t")
    .merge(
        df_cdc_prepared.alias("s"),
        "t.paciente_id = s.paciente_id"
    )
    .whenMatchedUpdate(
        condition="s.op = 4",  # update
        set={c: f"s.{c}" for c in target_cols}
    )
    .whenMatchedDelete(
        condition="s.op = 1"  # delete
    )
    .whenNotMatchedInsert(
        condition="s.op = 2",  # insert
        values={c: f"s.{c}" for c in target_cols}
    )
    .execute()
)




25/12/24 13:21:18 WARN MapPartitionsRDD: RDD 49 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

## 4. Obtener SOLO los registros afectados para Silver`

In [5]:
# IDs de filas afectadas en este lote
ids_cambiados = (
    df_cdc_prepared
    .select("paciente_id")
    .distinct()
)

ids_cambiados.show()

bronze_table="s3a://hospital/bronze/pacientes"

df_bronze = spark.read.format("delta").load(bronze_table)

# Hacer el join con los IDs que han cambiado
df_bronze_changed = (
    df_bronze
         .join(ids_cambiados, "paciente_id", "inner")
)



+-----------+
|paciente_id|
+-----------+
|         10|
|         11|
+-----------+



## 5. Aplicar REGLAS DE CALIDAD para Silver

In [6]:

df_silver_clean  = df_bronze_changed.filter(
    ((df_bronze_changed.email.isNotNull()) & (df_bronze_changed.email.contains("@"))) & ((df_bronze_changed.edad >= 0) & (df_bronze_changed.edad <= 120)) & (df_bronze_changed.genero.isin('F', 'M'))
)

df_silver_clean.show()

+-----------+--------------+----+------+--------------------+--------------------+
|paciente_id|        nombre|edad|genero|               email|     historia_medica|
+-----------+--------------+----+------+--------------------+--------------------+
|         10|  Laura García|  20|     M|laura.garcia@exam...|        Hipertensión|
|         11|Marta Martínez|  20|     M|marta.martinez@ex...|Hipertensión y ob...|
+-----------+--------------+----+------+--------------------+--------------------+



## 6. MERGE INCREMENTAL en Silver

In [7]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# Ruta de la tabla Delta en S3
silver_table_path = "s3a://hospital/silver/pacientes"

# Cargar la tabla Delta desde la ruta
delta_silver = DeltaTable.forPath(spark, silver_table_path)

# Columnas a actualizar/insertar
silver_cols = df_silver_clean.columns

# Crear los diccionarios para update e insert
update_set_silver = {c: F.col(f"s.{c}") for c in silver_cols}
insert_values_silver = {c: F.col(f"s.{c}") for c in silver_cols}

# Hacer el MERGE
(
    delta_silver.alias("t")
    .merge(
        df_silver_clean.alias("s"),
        "t.paciente_id = s.paciente_id"
    )
    .whenMatchedUpdate(set=update_set_silver)
    .whenNotMatchedInsert(values=insert_values_silver)
    .execute()
)


25/12/24 13:21:35 WARN MapPartitionsRDD: RDD 156 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

## 7. Actualizar el checkpoint con el último LSN procesado

In [8]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

df_cdc_prepared.select("start_lsn").distinct().show()

new_lsn = df_cdc_prepared.agg({"start_lsn": "max"}).first()[0]

if new_lsn is not None:
    new_lsn_bytes = bytes(new_lsn)
    # Crear un DataFrame con el nuevo LSN
    new_lsn_df = spark.createDataFrame([(new_lsn_bytes,)], "last_lsn binary")
    # Guardar como tabla Delta (append o overwrite según necesites)
    new_lsn_df.write.format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .save("s3a://hospital/bronze/pacientes_checkpoint")


new_lsn_df .show()

+--------------------+
|           start_lsn|
+--------------------+
|[00 00 00 33 00 0...|
|[00 00 00 33 00 0...|
+--------------------+



+--------------------+
|            last_lsn|
+--------------------+
|[00 00 00 33 00 0...|
+--------------------+



# Cerrar la sesión

In [9]:
spark.stop()